# Phaseformer on Kaggle: UIEB paired restoration
Attach the repository and UIEB dataset shown in your screenshot. Enable GPU and internet if a dependency needs installation. Run cells in order. For an inexpensive end-to-end check choose 2 epochs in cell 2; keep the same budget for both models. Full training uses 200 epochs and may span sessions. Save or download the experiment folder before ending a session; last.pt only protects completed epochs. Restore both best.pt and last.pt plus the logs and split using RESTORE. Keep the dataset mount and configuration unchanged.

**Report protocol:** New 720/80/90 split, seed 2025; model seed 123; best validation PSNR; low-resolution head at 256×256. Compare trained models on identical held-out images. Positive paired deltas favor NPS; a confidence interval crossing zero is inconclusive for that metric. Intervals describe test-image variation, not training-seed variation. UIQM uses the repository compatibility implementation. Conv/matmul FLOPs are partial; FFT work is separately estimated. The random-weight benchmark measures architecture latency, not image quality. Record the GPU from benchmark.json. Report pilot results as pilot results; published numbers were measured under a different protocol.

In [ ]:
# 1. Discover the two attached inputs. Keep Kaggle's GPU PyTorch installation.
from pathlib import Path
import os, sys, subprocess, json, zipfile, shutil, importlib.util
import torch
I, W = Path('/kaggle/input'), Path('/kaggle/working')
assert torch.cuda.is_available(), "Select a GPU accelerator, restart, then rerun."
print(torch.cuda.get_device_name(0), torch.__version__)
def one(paths, label):
    paths = list(paths)
    if len(paths) != 1: raise ValueError(f"Expected one {label}; found {paths}")
    return paths[0]
DATA = one((p.parent for p in I.rglob('raw-890') if (p.parent/'reference-890').is_dir()), 'paired UIEB root')
def repos(root):
    return [p.parent for p in root.rglob('train.py') if (p.parent/'configs/controlled.json').is_file() and (p.parent/'phaseformer_ext').is_dir()]
roots = repos(I)
if not roots:
    archive = one((p for p in I.rglob('*.zip') if 'phaseformer' in p.name.lower()), 'Phaseformer source ZIP')
    dest = W/'phaseformer_source'
    with zipfile.ZipFile(archive) as z: z.extractall(dest)
    roots = repos(dest)
REPO = one(roots, 'research repository')
print("Code:", REPO, "\nUIEB:", DATA)
missing = [pkg for mod,pkg in [('pytorch_msssim','pytorch-msssim==1.0.0'),('skimage','scikit-image'),('scipy','scipy'),('torchvision','torchvision')] if importlib.util.find_spec(mod) is None]
if missing: subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
def run(script,*args):
    subprocess.run([sys.executable,'-u',str(REPO/script),*map(str,args)],cwd=REPO,check=True,
                   env={**os.environ,'CUBLAS_WORKSPACE_CONFIG':':4096:8'})


In [ ]:
# 2. Choose the budget BEFORE training. 200 is the supplied full configuration.
# For a short pilot set EPOCHS=2; this creates a separate experiment, not a resumable prefix of 200.
EPOCHS, SEED = 200, 123
OUT = W/f'phaseformer_e{EPOCHS}_s{SEED}'
# To resume across sessions, attach your previously saved output and set RESTORE to the
# directory containing uieb.json and the baseline/nps run folder(s). Restore before preparing the split.
RESTORE = None
if RESTORE and not OUT.exists(): shutil.copytree(RESTORE,OUT)
OUT.mkdir(exist_ok=True)
M = OUT/'uieb.json'
if not M.exists(): run('scripts/prepare_data.py','--root',DATA,'--out',M,'--seed',2025,'--val-count',80,'--test-count',90)
doc=json.loads(M.read_text())
counts={k:len(v) for k,v in doc['splits'].items()}
print(counts)
assert counts=={'train':720,'val':80,'test':90} and Path(doc['root'])==DATA, 'Check the UIEB mount and saved split.'
(OUT/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
def train(v):
    folder=OUT/v
    last=folder/'last.pt'
    run('train.py','--config','configs/controlled.json','--manifest',M,'--variant',v,'--epochs',EPOCHS,
        '--seed',SEED,'--device','cuda','--output',folder,*(['--resume',last] if last.exists() else []))


In [ ]:
# 3. Original architecture, trained from scratch. Progress prints after each epoch.
train('baseline')


In [ ]:
# 4. Proposed architecture under the same settings.
train('nps')


In [ ]:
# 5. Held-out paired inference on both validation-selected checkpoints.
import pandas as pd
variants=['baseline','nps']
for v in variants:
    log=pd.read_json(OUT/v/'train.jsonl',lines=True)
    assert log.epoch.max()+1==EPOCHS, f'Finish training {v} before testing'
    run('evaluate.py','--manifest',M,'--split','test','--checkpoint',OUT/v/'best.pt','--output',OUT/v/'test','--device','cuda')
run('scripts/compare_results.py','--baseline',OUT/'baseline/test','--proposed',OUT/'nps/test','--out',OUT/'comparison.json')


In [ ]:
# 6. Fixed-shape single-GPU inference benchmark and report tables.
run('benchmark.py','--variants',*variants,'--size',256,'--device','cuda','--warmup',20,'--runs',100,'--out',OUT/'benchmark.json')
bench=json.loads((OUT/'benchmark.json').read_text())
rows=[]
for b in bench['rows']:
    v=b['variant']
    metrics=json.loads((OUT/v/'test/summary.json').read_text())['metrics']
    rows.append({**{k:b[k] for k in ['variant','parameters_registered','conv_matmul_gflops','fft_gflops_estimate','latency_ms_median','images_per_second']},**metrics})
table=pd.DataFrame(rows)
ci=pd.DataFrame(json.loads((OUT/'comparison.json').read_text())['metrics']).T
table.to_csv(OUT/'report_table.csv',index=False); ci.to_csv(OUT/'paired_differences.csv')
display(table,ci)


In [ ]:
# 7. Report figures: same four prespecified test examples for both models.
from PIL import Image
import matplotlib.pyplot as plt
fig, axes=plt.subplots(4,4,figsize=(10,10))
for axrow,row in zip(axes,doc['splits']['test'][:4]):
    name=row['name']
    for ax,p,title in zip(axrow,[DATA/row['input'],OUT/'baseline/test'/f'{name}.png',OUT/'nps/test'/f'{name}.png',DATA/row['target']],['Input','Baseline','NPS','Reference']):
        with Image.open(p) as im: ax.imshow(im.convert('RGB').resize((256,256),Image.Resampling.BICUBIC))
        ax.set_title(f'{title}: {name}',fontsize=8); ax.axis('off')
fig.tight_layout();fig.savefig(OUT/'examples.png',dpi=160);plt.show()
fig,axs=plt.subplots(1,2,figsize=(10,4))
for v in variants:
    logs=pd.read_json(OUT/v/'train.jsonl',lines=True)
    for ax,key in zip(axs,['train_loss','val_psnr_float_rgb']):
        ax.plot(logs.epoch+1,logs[key],label=v); ax.set(xlabel='Epoch',ylabel=key); ax.legend()
fig.tight_layout();fig.savefig(OUT/'learning_curves.png',dpi=150);plt.show()


In [ ]:
# 8. Optional: restore your own images, without reference targets.
NEW_IMAGES=None  # e.g. '/kaggle/input/my-underwater-photos/images' containing JPG or PNG files
if NEW_IMAGES:
    run('evaluate.py','--inputs',NEW_IMAGES,'--checkpoint',OUT/'nps/best.pt','--device','cuda','--output',OUT/'new_images')


In [ ]:
# 9. Download this archive or preserve the output folder for later resumption.
print(shutil.make_archive(str(W/(OUT.name+'_results')), 'zip', root_dir=OUT.parent,base_dir=OUT.name))
